# 01 — Type hints modernes

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- écrire des annotations modernes : `int | None`, `list[T]`, `dict[K, V]`
- annoter fonctions et méthodes pour `mypy --strict`
- utiliser `Callable`, `Iterable`, `Sequence`, `Mapping`
- manipuler `Any`, `object` et savoir quand les utiliser
- lire et comprendre une signature typée complexe

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- modèle objet complet (classes, héritage, ABC, Protocol)
- type hints de base vus en Initiation (`int | None`, `list[T]`)
- fonctions typées, f-strings

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- `Protocol` approfondi, `TypedDict`, `Literal`, `Final` (notebook 02)
- `Generic[T]`, `TypeVar`, PEP 695 (notebook 03)

## Plan

1. Rappel : pourquoi typer
2. Les primitives et `None`
3. Types génériques paramétrés
4. Unions avec `|` (PEP 604)
5. `Callable`, `Iterable`, `Sequence`, `Mapping`
6. `Any`, `object`, `Never`
7. Alias de type
8. Lancer `mypy` en démo
9. Synthèse
10. Exercices

---

## 1. Rappel : pourquoi typer

Le type hinting est **optionnel à l'exécution** mais essentiel pour :

- **documenter** : lire une signature donne l'intention ;
- **fiabiliser** : `mypy` (ou `pyright`) attrape des bugs avant le runtime ;
- **aider l'IDE** : autocomplétion, refactoring, navigation.

In [ ]:
def tarif(prix_ht: float, tva: float = 0.20) -> float:
    return prix_ht * (1 + tva)


In [ ]:
tarif(100.0)


---

## 2. Les primitives et `None`

Les types primitifs s'annotent directement avec le nom du type.

In [ ]:
age: int = 42
pi: float = 3.14
nom: str = 'Alice'
actif: bool = True


`None` s'annote `None`. Pour dire « `int` ou `None` », on écrit `int | None` (depuis **PEP 604**, Python 3.10+).

In [ ]:
def chercher(id_: int) -> str | None:
    if id_ == 1:
        return 'Alice'
    return None


In [ ]:
chercher(1), chercher(2)


---

## 3. Types génériques paramétrés

Depuis **PEP 585** (3.9+), on utilise les types **builtins** minuscules : `list[T]`, `dict[K, V]`, `tuple[T, ...]`, `set[T]`. On n'importe plus `List`, `Dict` de `typing`.

In [ ]:
def noms(utilisateurs: list[dict[str, str]]) -> list[str]:
    return [u['nom'] for u in utilisateurs]


In [ ]:
noms([{'nom': 'Alice'}, {'nom': 'Bob'}])


### `tuple` : homogène vs hétérogène

In [ ]:
coordonnees: tuple[float, float] = (3.14, 2.72)        # hétérogène fixe
chemin: tuple[int, ...] = (1, 2, 3, 4, 5)              # homogène de longueur variable


---

## 4. Unions avec `|` (PEP 604)

La syntaxe moderne des unions utilise le **pipe** `|`. Plus besoin d'importer `Union` ni `Optional`.

In [ ]:
def parse(valeur: str | int) -> int:
    if isinstance(valeur, int):
        return valeur
    return int(valeur)


In [ ]:
parse('42'), parse(42)


**Équivalences historiques :**

- `int | None` remplace `Optional[int]` et `Union[int, None]`.
- `int | str | float` remplace `Union[int, str, float]`.

`Optional[X]` reste valide mais n'est plus recommandé en code neuf.

---

## 5. `Callable`, `Iterable`, `Sequence`, `Mapping`

Pour typer des **comportements** plutôt que des classes concrètes, on utilise `collections.abc`. C'est l'application du principe *accepte large, renvoie étroit*.

In [ ]:
from collections.abc import Callable, Iterable, Sequence, Mapping


In [ ]:
def applique(f: Callable[[int], int], values: Iterable[int]) -> list[int]:
    return [f(v) for v in values]


In [ ]:
applique(lambda x: x * 2, [1, 2, 3])


**Lire `Callable[[int], int]`** : une fonction qui prend **un `int`** (c'est la liste) et renvoie un `int`.

In [ ]:
def somme_ponderee(valeurs: Sequence[float], poids: Sequence[float]) -> float:
    return sum(v * p for v, p in zip(valeurs, poids))


In [ ]:
somme_ponderee([10.0, 20.0, 30.0], [0.5, 0.3, 0.2])


### `Sequence` vs `list`

- `list[int]` : **uniquement** des listes.
- `Sequence[int]` : liste **ou** tuple **ou** toute séquence indexable et de longueur connue.

**Règle** : en paramètre d'entrée, préférez `Sequence`, `Iterable`, `Mapping` (plus permissif). En retour, soyez **précis** (retournez une `list[int]` concrète).

---

## 6. `Any`, `object`, `Never`

Trois cas limites à connaître.

- **`Any`** : désactive le typage. mypy accepte tout. À réserver aux interfaces avec du code non typé. **À utiliser avec parcimonie**.
- **`object`** : racine de toutes les classes. N'accepte que les opérations communes à **tout** objet.
- **`Never`** (PEP 484) : type sans valeur — retour d'une fonction qui **ne retourne jamais** (lève toujours, boucle infinie).

In [ ]:
from typing import Any, Never

def fail(raison: str) -> Never:
    raise RuntimeError(raison)


In [ ]:
def loose(x: Any) -> Any:
    return x.quoi_que_ce_soit  # mypy dit OK — vous n'avez aucune garantie


---

## 7. Alias de type

Donner un nom à un type composite. Deux syntaxes possibles.

In [ ]:
# Syntaxe classique (3.9+) :
Utilisateur = dict[str, str | int]

def afficher(u: Utilisateur) -> None:
    print(u['nom'])


In [ ]:
# Syntaxe PEP 695 (3.12+) — à privilégier :
type Utilisateur = dict[str, str | int]

def afficher2(u: Utilisateur) -> None:
    print(u['nom'])


---

---

## Synthèse

| Annotation | Sens |
|---|---|
| `int`, `float`, `str`, `bool` | Primitives |
| `int \| None` | Entier ou rien (PEP 604) |
| `list[T]`, `dict[K, V]`, `set[T]` | Génériques builtins (PEP 585) |
| `tuple[int, str]` | Tuple hétérogène fixe |
| `tuple[int, ...]` | Tuple homogène variable |
| `Callable[[int, str], bool]` | Fonction `(int, str) → bool` |
| `Iterable[T]`, `Sequence[T]`, `Mapping[K, V]` | Interfaces de `collections.abc` |
| `Any`, `object`, `Never` | Cas limites |
| `type Alias = ...` | Alias de type (PEP 695) |


### Règles à retenir

1. **Annotations modernes uniquement** : `int | None`, `list[T]`, pas de `Optional`, pas de `List`.
2. **Accepte large, renvoie étroit** : `Sequence` en entrée, `list` concrète en sortie.
3. **`Any` est une zone de désactivation**. Tout `Any` est une dette technique.
4. **`mypy --strict`** pour le code neuf. `pyright` pour la vitesse.
5. **Alias PEP 695** (`type X = ...`) pour les types composites réutilisés.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Annoter une fonction *(facile)*

Écrire une fonction `moyenne(valeurs: Sequence[float]) -> float | None` qui renvoie la moyenne ou `None` si la séquence est vide. Accepte aussi bien une `list` qu'un `tuple`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Type_hints_modernes", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from collections.abc import Sequence

def moyenne(valeurs: Sequence[float]) -> float | None:
    if not valeurs:
        return None
    return sum(valeurs) / len(valeurs)

print(moyenne([1.0, 2.0, 3.0]))
print(moyenne((10.0, 20.0)))
print(moyenne([]))
```

</details>

### Exercice 2 — Typer un dict de config *(facile)*

Écrire une fonction `ouvrir_port(config: dict[str, int | str | bool]) -> str` qui lit les clés `host` (str), `port` (int), `secure` (bool) et renvoie `'https://host:port'` ou `'http://host:port'` selon `secure`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Type_hints_modernes", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
def ouvrir_port(config: dict[str, int | str | bool]) -> str:
    host = config['host']
    port = config['port']
    secure = config['secure']
    schema = 'https' if secure else 'http'
    return f'{schema}://{host}:{port}'

print(ouvrir_port({'host': 'api.ex.com', 'port': 443, 'secure': True}))
```

</details>

### Exercice 3 — `Callable` en paramètre *(moyen)*

Écrire une fonction `appliquer_plusieurs(f: Callable[[int], int], valeurs: Iterable[int]) -> list[int]` qui applique `f` à chaque élément. Tester avec `lambda x: x ** 2` sur `range(1, 6)`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Type_hints_modernes", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from collections.abc import Callable, Iterable

def appliquer_plusieurs(f: Callable[[int], int], valeurs: Iterable[int]) -> list[int]:
    return [f(v) for v in valeurs]

print(appliquer_plusieurs(lambda x: x ** 2, range(1, 6)))
```

</details>

### Exercice 4 — Alias PEP 695 *(moyen)*

Définir un alias `type Coord = tuple[float, float]`. Écrire une fonction `distance(a: Coord, b: Coord) -> float`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Type_hints_modernes", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import math

type Coord = tuple[float, float]

def distance(a: Coord, b: Coord) -> float:
    return math.hypot(a[0] - b[0], a[1] - b[1])

print(distance((0, 0), (3, 4)))
```

</details>

---

## Ressources externes

### Documentation officielle
- [`typing` — stdlib](https://docs.python.org/3/library/typing.html)
- [`collections.abc`](https://docs.python.org/3/library/collections.abc.html)
- [mypy documentation](https://mypy.readthedocs.io/)

### PEPs de référence
- **PEP 484** — Type Hints (fondation)
- **PEP 585** — Generic builtins (`list[T]` au lieu de `List[T]`)
- **PEP 604** — Union avec `|`
- **PEP 695** — Type aliases et paramètres génériques (3.12+)